# backward-fn-signature — worked example 2: Write reciprocal_back reusing the cached out

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `backward-fn-signature`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

For `out = 1/x` the derivative is `d(out)/dx = -1/x**2`. Since `out = 1/x`, we have `out**2 = 1/x**2`, so `d(out)/dx = -out**2`. The chain rule gives `dL/dx = -grad_out * out**2`. Like `exp_back`, this op is most cheaply written in terms of the cached forward output rather than the raw input.

## Worked solution

**Step 1 — forward op.** `out = reciprocal(x) = 1/x`.

**Step 2 — local derivative.** `d(1/x)/dx = -x**(-2) = -1/x**2`.

**Step 3 — rewrite using out.** Because `out = 1/x`, squaring gives `out**2 = 1/x**2`. Therefore `d(out)/dx = -out**2`. Using `out` is both faster (no extra division) and numerically the same as `-1/x**2`.

**Step 4 — chain rule.** `dL/dx = grad_out * d(out)/dx = -grad_out * out**2`.

**Step 5 — signature & shape.** The fn takes the uniform `(grad_out, out, x)` triple; here `x` is unused. The result is elementwise, so it has the same shape and dtype as `x`. Note `out` carries everything we need — a clean illustration of why the cached output is part of the contract.

In [ ]:
def reciprocal_back(grad_out, out, x):
    # out = 1/x; d(out)/dx = -1/x**2 = -out**2. dL/dx = -grad_out * out**2.
    return -grad_out * out ** 2

t.manual_seed(0)
x = t.rand(5) + 0.5            # avoid divide-by-zero
out = 1.0 / x
grad_out = t.ones_like(out)
grad_x = reciprocal_back(grad_out, out, x)
print(grad_x)
print('matches -1/x^2:', t.allclose(grad_x, -1 / x ** 2))